# Comparison of training methods

Trains backprop, SigProp, DFA, SHARP and (optionally) Forward-Forward one after another in a single process, then compares them. Running sequentially keeps timing and memory measurements independent.

Every method gets the same data, model config and seed (so the same initialization and training batches) and is evaluated on the same held-out windows. Each run is saved to `runs_tinystories_*/`, and a summary of the whole batch goes to `runs_all/<timestamp>/`. A method that fails is logged and skipped.

An interrupted batch can be reloaded with `load_batch("runs_all/<timestamp>")`.

In [1]:
from ff_llm_lib.tinystories_data import load_stories
from ff_llm_lib.tokenizer import build_tokenizer
from ff_llm_lib.text_data import train_val_split
from ff_llm_lib.pipeline import setup
from ff_llm_lib.run_all import run_all, load_batch, summary_table, plot_comparison, plot_exits

SEED = 42

DATA_CONFIG = {
    "path": "train.csv",
    "fraction": 0.04,
    "train_frac": 0.9,
}
TOKENIZER_CONFIG = {"kind": "char"}
MODEL_CONFIG = {
    "block_size": 256,
    "n_embd": 256,
    "n_head": 8,
    "n_layer": 6,
    "dropout": 0.1,
    "use_rope": True,
    "use_rmsnorm": True,
    "use_swiglu": True,
}

# shared by all methods; per-method overrides go in METHODS[i]["train"]
COMMON_TRAIN = {
    "steps": 2000,
    "batch_size": 64,
    "warmup_steps": 200,
    "weight_decay": 0.01,
    "grad_clip": 1.0,
    "eval_every": 200,
}
FINAL_EVAL_WINDOWS = 256  # windows of block_size tokens

# run order = list order
METHODS = [
    {"name": "backprop", "kind": "backprop", "train": {"max_lr": 3e-3}},
    {"name": "sigprop", "kind": "sigprop", "train": {"max_lr": 3e-3, "eval_windows": 64}},
    {"name": "dfa", "kind": "dfa", "model": {"shared_feedback": False},
     "train": {"max_lr": 3e-3, "eval_windows": 64}},
    {"name": "sharp_G1_b0.5", "kind": "sharp", "model": {"group_size": 1, "beta": 0.5},
     "train": {"max_lr": 3e-3, "eval_windows": 64}},
    {"name": "sharp_G2_b0.5", "kind": "sharp", "model": {"group_size": 2, "beta": 0.5},
     "train": {"max_lr": 3e-3, "eval_windows": 64}},
    # ablations
    {"name": "sharp_G1_b0", "kind": "sharp", "model": {"group_size": 1, "beta": 0.0},
     "train": {"max_lr": 3e-3, "eval_windows": 64}, "enabled": False},
    {"name": "sharp_G1_b1", "kind": "sharp", "model": {"group_size": 1, "beta": 1.0},
     "train": {"max_lr": 3e-3, "eval_windows": 64}, "enabled": False},
    {"name": "sharp_G3_b0.5", "kind": "sharp", "model": {"group_size": 3, "beta": 0.5},
     "train": {"max_lr": 3e-3, "eval_windows": 64}, "enabled": False},
    {"name": "dfa_sharedB", "kind": "dfa", "model": {"shared_feedback": True},
     "train": {"max_lr": 3e-3, "eval_windows": 64}, "enabled": False},
    # Forward-Forward: much slower than the others, disabled by default
    {"name": "ff", "kind": "ff", "model": {"alpha": 1.0},
     "train": {"max_lr": 2e-3, "neg_k": 2, "n_mine": 0, "loss_kind": "nce", "eval_windows": 16},
     "final_eval_windows": 32, "enabled": False},
]

## Setup and data

In [2]:
device, _, _ = setup(SEED)

text = load_stories(DATA_CONFIG["path"], fraction=DATA_CONFIG["fraction"])
tokenizer = build_tokenizer(text, TOKENIZER_CONFIG)
data = tokenizer.encode(text)
train_data, val_data = train_val_split(data, train_frac=DATA_CONFIG["train_frac"])
MODEL_CONFIG["vocab_size"] = tokenizer.vocab_size

print(f"tokenizer: {TOKENIZER_CONFIG['kind']}, vocab_size={tokenizer.vocab_size}")
print(f"train={len(train_data):,} val={len(val_data):,} tokens")
print("will run:", [m["name"] for m in METHODS if m.get("enabled", True)])

device: mps
{'device': 'mps', 'platform': 'macOS-26.6.2-arm64-arm-64bit-Mach-O', 'cpu_count': 10, 'process_cpu_percent': 0.2, 'process_rss_mb': 272.46875, 'system_ram_used_gb': 9.348983764648438, 'system_ram_total_gb': 32.0, 'accelerator_mem_mb': 0.0}
tokenizer: char, vocab_size=99
train=70,170,259 val=7,796,696 tokens
will run: ['backprop', 'sigprop', 'dfa', 'sharp_G1_b0.5', 'sharp_G2_b0.5']


## Training

In [3]:
results, batch_dir = run_all(METHODS, MODEL_CONFIG, train_data, val_data, tokenizer, device, COMMON_TRAIN,
                             seed=SEED, final_eval_windows=FINAL_EVAL_WINDOWS)

batch dir: runs_all/20260927_171705

backprop  (backprop)  model={}  train={'steps': 2000, 'batch_size': 64, 'warmup_steps': 200, 'weight_decay': 0.01, 'grad_clip': 1.0, 'eval_every': 200, 'max_lr': 0.003}
parameters: 4,760,056
step      0 | train loss 4.5413 | dev loss 4.4676 | lr 0.00003


KeyboardInterrupt: 

## Comparison

To compare an earlier batch: `results = load_batch("runs_all/<timestamp>")`.

In [ ]:
summary_table(results)

In [ ]:
plot_comparison(results)

Validation cross-entropy at every exit (SigProp's per-block heads, SHARP's shared-readout exits), with backprop's full-depth value for reference.

In [ ]:
plot_exits(results)

## SHARP: early exit and self-speculative decoding

In [ ]:
for r in results:
    if r["kind"] != "sharp":
        continue
    print(f"== {r['name']}")
    for e in r["early_exit"]:
        print(f"  early exit thr {e['threshold']:.1f}: CE {e['cross_entropy']:.4f} | acc {e['accuracy']:.1%} "
              f"| compute {e['compute_fraction']:.0%}")
    for s in r["speculative"]:
        print(f"  speculative draft exit {s['draft_exit']}, n_draft {s['n_draft']}: accept {s['acceptance_rate']:.0%} "
              f"| block compute {s['relative_block_compute']:.2f}x | wall-clock speedup {s['speedup']:.2f}x")

## Samples

In [ ]:
for r in results:
    print(f"== {r['name']}  (val CE {r['val_ce']:.4f})")
    for s in r["samples"]:
        print("  ", s.replace("\n", " / "))
    print()